![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/Images/Noteable%20NB%20Header%20Banner.png)

# From Pixels to Process: An Interactive Burn Severity Mapping Notebook

A visually rich Earth observation workflow using real Landsat data from the Cold Springs Fire in Colorado, USA.

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

## 1. Setting Up Our Toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Import the raster, mapping, and visualisation tools we need for a real Earth observation workflow.
</div>

Click the code cell below and press the **<code>▶</code> Run** button in the toolbar (or use `Shift + Enter`). The progress messages will help you see exactly where we are in the analysis.

In [ ]:
print("Starting setup: importing raster, EO, and visualisation libraries...")

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="earthpy")


import re
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import xarray as xr
import earthpy as et
import matplotlib.pyplot as plt
import plotly.express as px
import folium
import ipywidgets as widgets

from IPython.display import display
from rasterio.plot import plotting_extent
from rasterio.transform import from_bounds
from rasterio.warp import transform_bounds, reproject, Resampling
from matplotlib.colors import ListedColormap, BoundaryNorm
from skimage.filters import threshold_otsu

plt.style.use("seaborn-v0_8-white")

print("Success! Imports ready.")
print("We are set up for real raster analysis, change detection, and interactive exploration.")

## 2. The Earth Observation Question

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Why this matters:</b> Wildfire changes vegetation structure, soil exposure, surface moisture, and reflectance patterns. Satellite imagery lets us observe those changes quickly across an entire landscape, which is why remote sensing is so important in environmental GeoScience.
</div>

In this notebook, we will analyse a real **pre-fire and post-fire Landsat scene pair** from the **Cold Springs Fire**.

We will learn how to connect:
- raster metadata and georeferencing
- multispectral bands and false-colour display
- a vegetation and burn indicator called **NBR**
- change detection using **dNBR**
- threshold choice and classification sensitivity
- interactive interpretation of burned area

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Our plan:</b> Download a compact public dataset, inspect the raster properties, build a burn-response indicator from the Landsat bands, compare different classification rules, and then explore how the mapped burned area changes when we adjust the threshold.
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> After your first successful run, come back to the editable thresholds later in the notebook and test how sensitive the burned-area estimate is to your analytical choices.
</div>

## 3. Loading a Real Wildfire Response Dataset

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Retrieve a small but real raster dataset that is robust enough for classroom use and rich enough for meaningful Earth observation analysis.
</div>

We will use an example dataset distributed through the `earthpy` teaching ecosystem. It contains Landsat imagery collected before and after the Cold Springs Fire, along with supporting spatial files.

In [ ]:
print("Downloading a real wildfire response dataset and locating the Landsat scenes...")

dataset_name = "cold-springs-fire"
data_dir = Path(et.data.get_data(dataset_name))

# TARGET THE SPECIFIC LANDSAT SUBFOLDER inside cold-springs-fire
# Earthpy places the pre/post folders inside 'landsat-collect'
landsat_dir = data_dir / "landsat-collect"
if not landsat_dir.exists():
    landsat_dir = data_dir  # Fallback if the folder structure varies slightly

# Find all TIFF files recursively within the target folder
tif_files = list(landsat_dir.rglob("*.tif")) + list(landsat_dir.rglob("*.TIF"))
scene_lookup = {}

for tif in tif_files:
    name = tif.name.upper()
    parent = tif.parent
    
    # Ignore MODIS/other sensor data that might be sitting in root folders
    if "MOD09" in name or "MCD45" in name:
        continue
        
    # Match band 4 (Red), band 5 (NIR), and band 7 (SWIR2) 
    # Works for both "_B4.TIF" and "CROP_BAND4.TIF" style naming rules
    if "B4" in name or "BAND4" in name:
        scene_lookup.setdefault(parent, {})["red"] = tif
    elif "B5" in name or "BAND5" in name:
        scene_lookup.setdefault(parent, {})["nir"] = tif
    elif "B7" in name or "BAND7" in name:
        scene_lookup.setdefault(parent, {})["swir2"] = tif

scene_records = []
for scene_dir, bands in scene_lookup.items():
    if {"red", "nir", "swir2"}.issubset(bands.keys()):
        scene_records.append({"scene_dir": scene_dir, **bands})

# Sort folders so that chronologically pre-fire comes first, post-fire last
scene_records = sorted(scene_records, key=lambda item: item["scene_dir"].name)

# Enhanced Debug display if tracking still fails
if len(scene_records) < 2:
    print(f"\n--- ENHANCED DEBUG INFO ---")
    print(f"Targeting directory: {landsat_dir}")
    print(f"Total valid TIFF files tracked: {len(tif_files)}")
    print("Available folders inside target:")
    for p in set(f.parent for f in tif_files):
        print(f" - Subfolder: {p.name} (Contains {len(list(p.glob('*.tif')))} tifs)")
    raise ValueError(f"The dataset downloaded, but two complete Landsat scenes were not found. Found: {len(scene_records)}")

pre_scene = scene_records[0]
post_scene = scene_records[-1]

def scene_name_to_date(name):
    # Matches strings like 'LC08_L1TP_034032_20160707...' or direct date folders
    match = re.search(r"(20\d{6})", name)
    if match:
        return pd.to_datetime(match.group(1), format="%Y%m%d").strftime("%d %b %Y")
    return name

pre_date = scene_name_to_date(pre_scene["scene_dir"].name)
post_date = scene_name_to_date(post_scene["scene_dir"].name)

shape_files = list(data_dir.rglob("*.shp"))
fire_boundary_path = None
for shp in shape_files:
    lower = shp.name.lower()
    if "fire" in lower or "bound" in lower or "perim" in lower:
        fire_boundary_path = shp
        break

scene_table = pd.DataFrame({
    "Role": ["Pre-fire scene", "Post-fire scene"],
    "Date label": [pre_date, post_date],
    "Scene folder": [pre_scene["scene_dir"].name, post_scene["scene_dir"].name]
})

print(f"Data source confirmed: {data_dir}")
print("Two Landsat scenes located for pre- and post-fire comparison.")
display(scene_table)

if fire_boundary_path is not None:
    print(f"Vector boundary found: {fire_boundary_path.name}")
else:
    print("No separate boundary shapefile was found. We can still work safely with the raster footprint.")


## 4. Inspecting Metadata and Spatial Context

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Important concept:</b> Raster cells only become meaningful geographically when they are correctly georeferenced. We need to know the raster's coordinate reference system, pixel size, extent, and nodata handling before we trust any distance, area, or spatial overlay result.
</div>

For this kind of analysis, the key metadata questions are:
- What CRS is the raster using?
- How big is each pixel on the ground?
- What is the scene extent?
- Are there nodata values we need to mask out?

In [ ]:
print("Inspecting raster metadata and georeferencing...")

with rasterio.open(pre_scene["red"]) as src:
    raster_profile = src.profile.copy()
    raster_bounds = src.bounds
    raster_res = src.res
    raster_extent_ll = transform_bounds(src.crs, "EPSG:4326", *src.bounds, densify_pts=21)

minx, miny, maxx, maxy = raster_bounds
extent_wkt = f"POLYGON(({minx} {miny}, {maxx} {miny}, {maxx} {maxy}, {minx} {maxy}, {minx} {miny}))"
extent_polygon = gpd.GeoDataFrame(
    {"name": ["Raster extent"]},
    geometry=gpd.GeoSeries.from_wkt([extent_wkt], crs=raster_profile["crs"])
)

fire_boundary = None
if fire_boundary_path is not None:
    fire_boundary = gpd.read_file(fire_boundary_path)
    if fire_boundary.crs is None:
        fire_boundary = fire_boundary.set_crs(raster_profile["crs"])
    else:
        fire_boundary = fire_boundary.to_crs(raster_profile["crs"])

metadata_table = pd.DataFrame({
    "Property": [
        "CRS",
        "Raster size",
        "Pixel size",
        "Nodata value",
        "Bounds (projected)",
        "Bounds (lat/lon)"
    ],
    "Value": [
        str(raster_profile["crs"]),
        f"{raster_profile['width']} columns × {raster_profile['height']} rows",
        f"{raster_res[0]:.1f} m × {abs(raster_res[1]):.1f} m",
        str(raster_profile["nodata"]),
        f"{raster_bounds}",
        f"{tuple(round(v, 4) for v in raster_extent_ll)}"
    ]
})

display(metadata_table)

fig, ax = plt.subplots(figsize=(6, 6))
extent_polygon.boundary.plot(ax=ax, color="#252525", linewidth=1.5, label="Raster extent")
if fire_boundary is not None:
    fire_boundary.boundary.plot(ax=ax, color="#d7301f", linewidth=1.2, label="Boundary layer")
ax.set_title("Georeferenced study footprint", fontsize=13)
ax.legend(loc="lower left")
ax.set_axis_off()
plt.show()

print("Metadata inspected. We now know the raster resolution, extent, and coordinate system.")

## 5. From Bands to Burn Indicators

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Turn the raw Landsat bands into interpretable Earth observation products.
</div>

We will create three key outputs:

- a **false-colour composite** using SWIR2, NIR, and Red
- **NBR** for the pre-fire scene
- **NBR** for the post-fire scene
- **dNBR = pre-fire NBR − post-fire NBR**

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Interpretation hint:</b> In burn-response analysis, higher dNBR values often indicate stronger landscape change associated with vegetation loss and surface exposure. It is a useful proxy, but it is still a simplification of real ecological process.
</div>

In [ ]:
print("Loading bands, masking nodata, and building burn indicators...")

def read_landsat_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile.copy()
        nodata = src.nodata
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    arr = np.where(arr <= 0, np.nan, arr)
    return arr, profile

red_pre, raster_profile = read_landsat_band(pre_scene["red"])
nir_pre, _ = read_landsat_band(pre_scene["nir"])
swir_pre, _ = read_landsat_band(pre_scene["swir2"])

red_post, _ = read_landsat_band(post_scene["red"])
nir_post, _ = read_landsat_band(post_scene["nir"])
swir_post, _ = read_landsat_band(post_scene["swir2"])

if red_pre.shape != red_post.shape:
    raise ValueError("The pre-fire and post-fire scenes are not aligned. This notebook expects matching rasters.")

valid_mask = (
    np.isfinite(red_pre) & np.isfinite(nir_pre) & np.isfinite(swir_pre) &
    np.isfinite(red_post) & np.isfinite(nir_post) & np.isfinite(swir_post)
)

def safe_index(numerator, denominator):
    return np.divide(
        numerator,
        denominator,
        out=np.full_like(numerator, np.nan, dtype="float32"),
        where=np.abs(denominator) > 0
    )

nbr_pre = safe_index(nir_pre - swir_pre, nir_pre + swir_pre)
nbr_post = safe_index(nir_post - swir_post, nir_post + swir_post)
dnbr = nbr_pre - nbr_post
dnbr[~valid_mask] = np.nan

pixel_area_m2 = abs(raster_profile["transform"].a * raster_profile["transform"].e)
pixel_area_ha = pixel_area_m2 / 10000

def stretch_composite(r, g, b, low=2, high=98):
    stack = np.dstack([r, g, b]).astype("float32")
    out = np.zeros_like(stack, dtype="float32")
    for i in range(3):
        band = stack[:, :, i]
        lo, hi = np.nanpercentile(band, [low, high])
        if hi > lo:
            out[:, :, i] = (band - lo) / (hi - lo)
    return np.clip(out, 0, 1)

false_colour_pre = stretch_composite(swir_pre, nir_pre, red_pre)
false_colour_post = stretch_composite(swir_post, nir_post, red_post)

rows = np.arange(dnbr.shape[0])
cols = np.arange(dnbr.shape[1])
x_coords = raster_profile["transform"].c + (cols + 0.5) * raster_profile["transform"].a
y_coords = raster_profile["transform"].f + (rows + 0.5) * raster_profile["transform"].e

dnbr_xr = xr.DataArray(
    dnbr,
    dims=("y", "x"),
    coords={"y": y_coords, "x": x_coords},
    name="dNBR",
    attrs={
        "long_name": "Differenced Normalized Burn Ratio",
        "crs": str(raster_profile["crs"]),
        "pixel_size_m": abs(raster_profile["transform"].a)
    }
)

summary_table = pd.DataFrame({
    "Statistic": ["Mean pre-fire NBR", "Mean post-fire NBR", "Mean dNBR", "95th percentile dNBR", "Valid pixels"],
    "Value": [
        float(np.nanmean(nbr_pre)),
        float(np.nanmean(nbr_post)),
        float(dnbr_xr.mean(skipna=True).item()),
        float(dnbr_xr.quantile(0.95, skipna=True).item()),
        int(np.isfinite(dnbr).sum())
    ]
})
display(summary_table.round(3))

extent = plotting_extent(dnbr, raster_profile["transform"])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(false_colour_pre, extent=extent)
axes[0].set_title(f"False colour before fire\n{pre_date}", fontsize=13)

axes[1].imshow(false_colour_post, extent=extent)
axes[1].set_title(f"False colour after fire\n{post_date}", fontsize=13)

im = axes[2].imshow(dnbr, cmap="magma", vmin=-0.2, vmax=0.8, extent=extent)
axes[2].set_title("dNBR change surface", fontsize=13)

for ax in axes:
    if fire_boundary is not None:
        fire_boundary.boundary.plot(ax=ax, color="white", linewidth=0.8)
    ax.set_axis_off()

cbar = fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
cbar.set_label("dNBR")
plt.tight_layout()
plt.show()

print("Raster products prepared: false-colour views and dNBR are ready.")

## 6. Thresholding, Classification, and Comparison

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Why compare rules?</b> A threshold turns a continuous raster into a simpler map, but different thresholds can tell very different stories. That is why responsible Earth observation always checks sensitivity.
</div>

Here we compare:
- an **inclusive threshold** for a broad burned-area estimate
- a **manual severity cutoff** often used as a more conservative rule
- an **Otsu threshold** chosen from the image histogram itself

We will also build a five-class burn-severity style map from dNBR values.

In [ ]:
print("Comparing threshold rules and creating a burn severity classification...")

# EDIT THIS VALUE: a broad threshold that captures more changed pixels
inclusive_threshold = 0.10

# EDIT THIS VALUE: a more conservative threshold for mapping burned pixels
manual_threshold = 0.27

otsu_threshold = float(threshold_otsu(dnbr[np.isfinite(dnbr)]))

def burned_area_ha(threshold):
    burned_mask = np.isfinite(dnbr) & (dnbr >= threshold)
    return burned_mask.sum() * pixel_area_ha

comparison_df = pd.DataFrame({
    "Rule": [
        f"Inclusive threshold ({inclusive_threshold:.2f})",
        f"Manual severity cutoff ({manual_threshold:.2f})",
        f"Otsu data-driven threshold ({otsu_threshold:.2f})"
    ],
    "Threshold": [inclusive_threshold, manual_threshold, otsu_threshold],
    "Burned area (ha)": [
        burned_area_ha(inclusive_threshold),
        burned_area_ha(manual_threshold),
        burned_area_ha(otsu_threshold)
    ],
    "Burned share of valid pixels (%)": [
        100 * np.mean(dnbr[np.isfinite(dnbr)] >= inclusive_threshold),
        100 * np.mean(dnbr[np.isfinite(dnbr)] >= manual_threshold),
        100 * np.mean(dnbr[np.isfinite(dnbr)] >= otsu_threshold)
    ]
})
display(comparison_df.round(2))

severity_bins = [0.10, 0.27, 0.44, 0.66]
severity_labels = [
    "Unchanged / very low",
    "Low",
    "Moderate-low",
    "Moderate-high",
    "High"
]
severity_colors = ["#f7f7f7", "#fed976", "#fd8d3c", "#e6550d", "#a63603"]

severity_code = np.full(dnbr.shape, -1, dtype="int16")
severity_code[np.isfinite(dnbr)] = np.digitize(dnbr[np.isfinite(dnbr)], bins=severity_bins)

hist_values = dnbr[np.isfinite(dnbr)].ravel()

fig_hist = px.histogram(
    x=hist_values,
    nbins=70,
    title="dNBR distribution: where do the classification thresholds sit?",
    labels={"x": "dNBR", "y": "Pixel count"},
    color_discrete_sequence=["#6a51a3"]
)
for x_value, line_color, label in [
    (inclusive_threshold, "#31a354", f"Inclusive {inclusive_threshold:.2f}"),
    (manual_threshold, "#de2d26", f"Manual {manual_threshold:.2f}"),
    (otsu_threshold, "#3182bd", f"Otsu {otsu_threshold:.2f}")
]:
    fig_hist.add_vline(x=x_value, line_color=line_color, line_dash="dash")
    fig_hist.add_annotation(
        x=x_value,
        y=1,
        yref="paper",
        text=label,
        showarrow=False,
        xanchor="left",
        font=dict(color=line_color)
    )
fig_hist.update_layout(xaxis_title="dNBR", yaxis_title="Pixel count")

fig_bar = px.bar(
    comparison_df,
    x="Rule",
    y="Burned area (ha)",
    color="Rule",
    title="Mapped burned area changes when the rule changes",
    color_discrete_sequence=["#31a354", "#de2d26", "#3182bd"]
)
fig_bar.update_layout(showlegend=False, xaxis_title="", yaxis_title="Burned area (ha)")

fig_hist.show()
fig_bar.show()

from matplotlib.patches import Patch

cmap = ListedColormap(severity_colors)
norm = BoundaryNorm(np.arange(-0.5, 5.5, 1), cmap.N)

fig, ax = plt.subplots(figsize=(8, 8))
masked_classes = np.ma.masked_where(severity_code < 0, severity_code)
ax.imshow(masked_classes, cmap=cmap, norm=norm, extent=extent)
if fire_boundary is not None:
    fire_boundary.boundary.plot(ax=ax, color="#252525", linewidth=0.9)
ax.set_title("Five-class burn severity style map from dNBR", fontsize=14)
ax.set_axis_off()
legend_handles = [Patch(facecolor=severity_colors[i], edgecolor="#333333", label=severity_labels[i]) for i in range(len(severity_labels))]
ax.legend(handles=legend_handles, loc="lower left", frameon=True)
plt.show()

print("Threshold comparison complete.")

## 7. Interactive Threshold Explorer

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Explore threshold sensitivity directly. This is where the notebook moves from static mapping to process-oriented interpretation.
</div>

Run the next cell, then move the slider. Watch how a small change in the cut-off can produce a much larger or smaller burned-area estimate.

In [ ]:
print("Building an interactive threshold explorer...")

threshold_slider = widgets.FloatSlider(
    value=manual_threshold,
    min=0.05,
    max=0.70,
    step=0.01,
    description="dNBR cut-off:",
    continuous_update=False,
    readout_format=".2f",
    style={"description_width": "initial"}
)

binary_cmap = ListedColormap(["#f0f0f0", "#cb181d"])

def threshold_view(threshold):
    burned = np.isfinite(dnbr) & (dnbr >= threshold)
    area_ha = burned.sum() * pixel_area_ha
    share = 100 * burned.sum() / np.isfinite(dnbr).sum()
    mean_selected = float(np.nanmean(dnbr[burned])) if burned.any() else np.nan

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    im = axes[0].imshow(dnbr, cmap="magma", vmin=-0.2, vmax=0.8, extent=extent)
    if fire_boundary is not None:
        fire_boundary.boundary.plot(ax=axes[0], color="white", linewidth=0.7)
    axes[0].set_title("Continuous dNBR surface")
    axes[0].set_axis_off()
    cbar = plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
    cbar.set_label("dNBR")

    axes[1].imshow(np.where(burned, 1, 0), cmap=binary_cmap, extent=extent, vmin=0, vmax=1)
    if fire_boundary is not None:
        fire_boundary.boundary.plot(ax=axes[1], color="#252525", linewidth=0.7)
    axes[1].set_title(f"Pixels flagged as burned at threshold {threshold:.2f}")
    axes[1].set_axis_off()

    plt.show()

    summary = pd.DataFrame({
        "Measure": [
            "Threshold",
            "Mapped burned area (ha)",
            "Share of valid pixels (%)",
            "Mean dNBR of flagged pixels"
        ],
        "Value": [threshold, area_ha, share, mean_selected]
    })
    display(summary.round(3))

interactive_view = widgets.interactive_output(threshold_view, {"threshold": threshold_slider})

display(widgets.HTML(
    "<div style='border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:12px; margin:8px 0; border-radius:4px;'>"
    "<b>Try this:</b> Move the slider from 0.10 up toward 0.50. Notice how the map becomes more selective, and how the estimated burned area drops."
    "</div>"
))
display(threshold_slider, interactive_view)

print("Interactive explorer ready.")

## 8. Interactive Web Map of the Final Interpretation

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Map legend:</b> In the final overlay below, pale tones represent little change, while darker orange-brown tones represent stronger burn-response values in the dNBR classification.
</div>

This web map places the burn-severity style result on top of an imagery basemap so you can inspect the spatial pattern directly.

In [ ]:
print("Preparing a web map overlay for the final burn severity result...")

west, south, east, north = transform_bounds(raster_profile["crs"], "EPSG:4326", *raster_bounds, densify_pts=21)

dst_transform = from_bounds(west, south, east, north, raster_profile["width"], raster_profile["height"])
severity_ll = np.full_like(severity_code, -1)

reproject(
    source=severity_code,
    destination=severity_ll,
    src_transform=raster_profile["transform"],
    src_crs=raster_profile["crs"],
    dst_transform=dst_transform,
    dst_crs="EPSG:4326",
    src_nodata=-1,
    dst_nodata=-1,
    resampling=Resampling.nearest
)

rgba = np.zeros((severity_ll.shape[0], severity_ll.shape[1], 4), dtype=np.uint8)
rgba_palette = {
    0: (247, 247, 247, 20),
    1: (254, 217, 118, 180),
    2: (253, 141, 60, 180),
    3: (230, 85, 13, 190),
    4: (166, 54, 3, 210)
}
for class_id, color in rgba_palette.items():
    rgba[severity_ll == class_id] = color

web_map = folium.Map(
    location=[(south + north) / 2, (west + east) / 2],
    zoom_start=12,
    tiles=None
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite basemap"
).add_to(web_map)

folium.raster_layers.ImageOverlay(
    image=rgba,
    bounds=[[south, west], [north, east]],
    name="Burn severity overlay",
    opacity=0.75,
    interactive=True
).add_to(web_map)

outline_gdf = fire_boundary.to_crs(4326) if fire_boundary is not None else extent_polygon.to_crs(4326)
folium.GeoJson(
    outline_gdf.to_json(),
    name="Study outline",
    style_function=lambda feature: {
        "color": "#00ffff",
        "weight": 2,
        "fillOpacity": 0
    }
).add_to(web_map)

folium.LayerControl(collapsed=False).add_to(web_map)

print("Interactive web map ready.")
display(web_map)

## 9. Uncertainty, Resolution, and Responsible Interpretation

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Important limitations:</b> dNBR is powerful, but it is not a perfect description of ecological burn severity. It is a spectral change metric, not ground truth.
</div>

Here are the main cautions to keep in mind:

- **Threshold sensitivity matters.** Different rules can produce substantially different burned-area estimates.
- **Pixels are mixed.** At 30 m resolution, one pixel can contain vegetation, bare soil, rock, and shadow all at once.
- **Atmosphere and illumination matter.** Smoke, haze, terrain shadow, and acquisition differences can affect reflectance.
- **Time matters.** A post-fire image captures one moment after the event. Regrowth, delayed effects, and seasonal conditions are not fully represented.
- **Classification simplifies a continuum.** Real landscapes change gradually, but thresholding forces hard boundaries between classes.
- **Boundary files and rasters may not match perfectly.** Small alignment differences are normal when different data products are combined.

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Good scientific practice:</b> Treat this workflow as an evidence-building tool. In a fuller project, you would add field validation, compare more dates, examine terrain and weather context, and justify any thresholds with domain literature or local expertise.
</div>

## Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    Congratulations! You have completed a real raster-led Earth observation workflow. You loaded georeferenced multispectral imagery, inspected raster metadata, built false-colour views, calculated NBR and dNBR, compared classification rules, and explored threshold sensitivity interactively. This is exactly the kind of reproducible, process-aware remote sensing workflow that modern GeoScience depends on.
</div>

### Suggested next steps

- Compare a longer pre-fire and post-fire time series
- Add a DEM and explore how slope or aspect influences the pattern
- Test a different thresholding rule or a supervised classification approach
- Summarise burn classes by management zone or catchment boundary
- Compare this fire scar with another real event to see how context changes the result

If you edit any thresholds or parameters, re-run the notebook from top to bottom so every output stays in sync.

![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/Images/Noteable%20Notebook%20Footer.png)